In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.3 LoRA

Full fine-tuning updates every weight. **LoRA** freezes the base and trains a tiny
low-rank update (`B @ A`) on a few layers instead, a fraction of the parameters,
so you can adapt a model on modest hardware and keep many small adapters.

In [ ]:
from pocketlm import (train_tiny, pick_config, add_lora, trainable_parameters,
                      make_batch)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
total = sum(p.numel() for p in model.parameters())
add_lora(model, rank=4)
trainable = sum(p.numel() for p in trainable_parameters(model))
print(f"trainable: {trainable} / {total}  ({100 * trainable / total:.1f}%)")

data = torch.tensor(tok.encode(corpus), dtype=torch.long)
opt = torch.optim.AdamW(trainable_parameters(model), lr=1e-3)
g = torch.Generator().manual_seed(0)
first = None
for _ in range(60 if os.environ.get("POCKETLM_CI") else 150):
    x, y = make_batch(data, model.cfg.block_size, 16, generator=g)
    _, loss = model(x, y)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("adapter-only loss:", round(first, 3), "->", round(loss.item(), 3))

Only the adapter trains, yet the loss still moves. Swap adapters to switch tasks
without touching the (large, shared) base weights.

In [ ]:
assert 0 < trainable < total * 0.2
assert loss.item() <= first